# DSpace DB5 extract

Notebook de prueba para el nodo `extract_dspacedb5_tables` del pipeline `extract_dspacedb5`.

Lee tablas desde `dspacedb5/public/{table}`, agrega metadata de extraccion y deja los DataFrames listos para persistir como `raw/dspacedb5/{table}#parquet` en el pipeline. Esta notebook no hace `catalog.save`.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)


In [ ]:
source_label = catalog.load("params:extract_dspacedb5_options.source_label")
institution_ror = catalog.load("params:extract_dspacedb5_options.institution_ror")
extract_env = catalog.load("params:extract_dspacedb5_options.env")
filter_param = catalog.load("params:extract_dspacedb5_options.filter_param")
filter_value = catalog.load("params:extract_dspacedb5_options.filter_value")
tables = catalog.load("params:extract_dspacedb5_options.tables")

tables


In [ ]:
dataframes = [catalog.load(f"dspacedb5/public/{table}") for table in tables]

[(table, df.shape) for table, df in zip(tables, dataframes)]


In [ ]:
DSPACE_DB5_TABLES = (
    "bitstream",
    "bundle2bitstream",
    "collection2item",
    "collection",
    "community2collection",
    "community2community",
    "community",
    "handle",
    "item2bundle",
    "item",
    "metadatafieldregistry",
    "metadataschemaregistry",
    "metadatavalue",
)


def _add_extract_metadata(
    df: pd.DataFrame,
    *,
    source_label: str,
    institution_ror: str,
    extract_env: str,
    source_table: str,
    filter_param: str | None,
    filter_value,
    extract_datetime: pd.Timestamp,
) -> pd.DataFrame:
    enriched_df = df.copy()
    enriched_df["_source_system"] = "dspacedb5"
    enriched_df["_source_table"] = source_table
    enriched_df["_extract_datetime"] = extract_datetime
    enriched_df["_extract_date"] = extract_datetime.date()
    enriched_df["_source_label"] = source_label
    enriched_df["_institution_ror"] = institution_ror
    enriched_df["_extract_env"] = extract_env
    enriched_df["_filter_param"] = filter_param if filter_param else pd.NA
    enriched_df["_filter_value"] = filter_value if filter_value not in (None, "") else pd.NA
    return enriched_df


def extract_dspacedb5_tables(
    tables,
    source_label,
    institution_ror,
    extract_env,
    filter_param,
    filter_value,
    *dataframes,
):
    configured_tables = tuple(tables)

    if configured_tables != DSPACE_DB5_TABLES:
        raise ValueError(
            f"Configured tables do not match pipeline inputs. "
            f"Expected {DSPACE_DB5_TABLES}, got {configured_tables}."
        )

    if len(dataframes) != len(DSPACE_DB5_TABLES):
        raise ValueError(
            f"Expected {len(DSPACE_DB5_TABLES)} dataframes, got {len(dataframes)}."
        )

    extract_datetime = pd.Timestamp.now(tz="UTC").floor("s").tz_localize(None)

    return tuple(
        _add_extract_metadata(
            df,
            source_label=source_label,
            institution_ror=institution_ror,
            extract_env=extract_env,
            source_table=table_name,
            filter_param=filter_param,
            filter_value=filter_value,
            extract_datetime=extract_datetime,
        )
        for table_name, df in zip(DSPACE_DB5_TABLES, dataframes)
    )


In [ ]:
extracted_tables = extract_dspacedb5_tables(
    tables,
    source_label,
    institution_ror,
    extract_env,
    filter_param,
    filter_value,
    *dataframes,
)

summary = pd.DataFrame(
    {
        "table": tables,
        "rows": [len(df) for df in extracted_tables],
        "columns": [len(df.columns) for df in extracted_tables],
        "source_system": [df["_source_system"].iloc[0] if len(df) else "dspacedb5" for df in extracted_tables],
        "source_label": [df["_source_label"].iloc[0] if len(df) else source_label for df in extracted_tables],
        "institution_ror": [df["_institution_ror"].iloc[0] if len(df) else institution_ror for df in extracted_tables],
        "extract_datetime": [df["_extract_datetime"].iloc[0] if len(df) else pd.NA for df in extracted_tables],
    }
)

summary


## Inspeccion rapida

Usar `table_to_preview` para revisar una tabla puntual antes de copiar cambios a `nodes.py`.

In [ ]:
table_to_preview = "item"
df_preview = extracted_tables[tables.index(table_to_preview)]

df_preview.head()


In [ ]:
metadata_columns = [
    "_source_system",
    "_source_table",
    "_extract_datetime",
    "_extract_date",
    "_source_label",
    "_institution_ror",
    "_extract_env",
    "_filter_param",
    "_filter_value",
]

df_preview[metadata_columns].head()
